# 10 — Joint Switching Dynamic Factor HMM: Do Factor Dynamics Improve Price? (framework v3.2)

Module 09 established a stable factor basis and persistent non-switching factor dynamics, but direct factor → next-return regressions and the omega_F=0 factor-augmented switching regression did not improve on the simpler Notebook 08 benchmark.

Module 10 tests the central v3.2 hypothesis:

> the market measurements represent the same latent forces, but the dynamics of those forces change with the structural regime, and those changes are retained only if they improve chronological out-of-sample price prediction.

The exploratory joint model is

$$
F_{j,t}=c_{R_t,j}+a_{R_t,j}F_{j,t-1}+s_{R_t,j}\eta_{j,t},
$$

$$
z_{t+1}=\mu_{R_t}+\beta_{D,R_t}D_t+\gamma_{R_t}^\top F_t^{dir}+\rho^\top L_t+\sigma_{R_t}\xi_{t+1},
$$

with tempered factor evidence

$$
\ell_t(k)=\ell_t^{price}(k)+\omega_F\ell_t^{factor}(k).
$$

Deliberate restrictions:

- HMM only. HSMM duration is tested only if the joint HMM earns its place.
- K in {1,2}. This is not a search for MR/MOM/VOL/Bridge.
- Diagonal A_R baseline. Module 09 showed own-factor persistence was much more important than dense cross-factor propagation.
- State-specific factor drift/persistence is shrunk toward the non-switching factor dynamics.
- Missing factors are masked equation by equation; a missing option/state family never drops the full timestamp and is never converted into an economic zero.
- At time t the factor transition is observed first, updates P(R_t), and only then is z_{t+1} scored. This preserves causal timing.
- omega_F is selected using blocked CV inside the first-half training sessions only. The second-half holdout is untouched until the final comparison.

Primary decision: does the selected omega_F > 0 K=2 joint model beat the identical omega_F=0 K=2 price-switching benchmark on held-out sessions?

Status: discovery research. Approximately 15 sessions do not permit structural regime claims under v3.2.

In [1]:
from pathlib import Path
import os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 240)

load_dotenv()
DB_PATH = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve() / "database" / "research.duckdb"
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling.clock_baseline import ClockBaselineSet, FeatureSpec
from data_engineering.modeling import factors as fx
from data_engineering.modeling import sdf_hmm as sdf
from data_engineering.modeling.hsmm.design import build_design
from data_engineering.modeling.hsmm.selection import blocked_folds
from data_engineering.modeling.hsmm.session_eval import paired_sessions

if tuple(int(x) for x in ri.REGIME_INPUT_VERSION.split(".")) < (0, 3) or fx.FACTOR_VERSION < "0.2" or sdf.SDF_HMM_VERSION < "0.1.1":
    raise RuntimeError("Old src modules: git pull and restart the kernel.")

CONTRACT_ASSETS = ["BTC", "XRP"]  # corrective Module 10 rerun only; full six-asset results remain in prior commit
OMEGAS = tuple(float(x) for x in os.getenv("NB10_OMEGAS", "0,0.1,0.25,0.5,1").split(","))
N_BOOT = int(os.getenv("NB10_BOOT", "2000"))
MAX_ITER = int(os.getenv("NB10_MAX_ITER", "120"))
N_INIT = int(os.getenv("NB10_N_INIT", "3"))
SCREEN_MAX_ITER = int(os.getenv("NB10_SCREEN_MAX_ITER", "60"))
SCREEN_N_INIT = int(os.getenv("NB10_SCREEN_N_INIT", "1"))
FACTOR_SHRINK = float(os.getenv("NB10_FACTOR_SHRINK", "100"))
SCALE_SHRINK = float(os.getenv("NB10_SCALE_SHRINK", "50"))
print(f"src {SRC} | sdf_hmm {sdf.SDF_HMM_VERSION} | omega grid {OMEGAS} | factor shrink {FACTOR_SHRINK:g} | "
      f"screen {SCREEN_N_INIT} init × {SCREEN_MAX_ITER} iter | final {N_INIT} init × {MAX_ITER} iter")

src D:\Projects\deribit_market_data\src | sdf_hmm 0.1.1 | omega grid (0.0, 0.1, 0.25, 0.5, 1.0) | factor shrink 100 | screen 1 init × 60 iter | final 3 init × 120 iter


## 1. Load the v3.2 state table and freeze the outer train/test split

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])
base_cols = ["state_id", "state_time", "underlying", "session_id", "return_5m", "futures_index_price", "is_weekend",
             "in_settlement_window", "fit_ok", "option_atm_quote_ok",
             "vwap", "volume_usd", "trade_count", "futures_tradable_mid_price", "perp_integrity_ok",
             "known_exchange_event", "integrity_flag_trades_outside_bbo"]
factor_cols = sorted({fd.column for fam in fx.FAMILIES for fd in fam.features} |
                     {"trade_flow_imbalance", "book_twa_obi_20", "book_twa_bid_depth_20", "book_twa_ask_depth_20",
                      "option_iv_atm_forward", "realized_vol_1h_annualized"})
want = [c for c in dict.fromkeys(base_cols + factor_cols) if c in cols]
df = con.execute(f"SELECT {', '.join(want)} FROM time_series_feature_state ORDER BY underlying, state_time").df()
df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
for c in ["is_weekend", "in_settlement_window", "fit_ok", "option_atm_quote_ok", "perp_integrity_ok",
          "known_exchange_event", "integrity_flag_trades_outside_bbo"]:
    if c in df:
        s = df[c]
        if s.dtype == object:
            s = s.map({"True": True, "False": False, True: True, False: False})
        df[c] = s.fillna(False).astype(bool)
df = df[df["underlying"].isin(CONTRACT_ASSETS)].reset_index(drop=True)
df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["session_date"] = ri.session_key(df["state_time"])
dates = sorted(df.loc[df["r"].notna(), "session_date"].unique())

# Prospective freeze from the original Module 10 experiment. Newly downloaded
# sessions stay out of sample instead of moving the train/test boundary.
FROZEN_TRAIN_START = "2026-09-22"
FROZEN_TRAIN_END = "2026-09-28"
TRAIN_DATES = {d for d in dates if FROZEN_TRAIN_START <= str(d) <= FROZEN_TRAIN_END}
TEST_DATES = {d for d in dates if str(d) > FROZEN_TRAIN_END}
if not TRAIN_DATES or not TEST_DATES:
    raise RuntimeError("Frozen Module 10 split needs both training and post-2026-09-28 test sessions.")

print(f"Rows {len(df):,} | sessions {len(dates)} | frozen train {min(TRAIN_DATES)} → {max(TRAIN_DATES)} ({len(TRAIN_DATES)}) | "
      f"all later sessions held out {min(TEST_DATES)} → {max(TEST_DATES)} ({len(TEST_DATES)})")

Rows 10,368 | sessions 16 | frozen train 2026-09-22 → 2026-09-28 (7) | all later sessions held out 2026-09-29 → 2026-10-07 (9)


## 2. Freeze the factor basis on the outer training block

This repeats Module 09's training-only construction. Families must pass the same coverage/loading-stability gate. The final holdout does not affect clock normalization, loadings, factor eligibility, omega_F, or fitted parameters.

In [3]:
def build_context(fit_dates):
    mask = df["session_date"].isin(set(fit_dates))
    clock = ClockBaselineSet.fit(
        df[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]], mask,
        specs=(FeatureSpec("return_5m", "return_5m", transform="log_abs", scale="none"),)
    )
    f_clock = clock.transform(
        df[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]]
    )["return_5m__clock_vol_factor"].fillna(1.0)

    FS = fx.build_factors(df, mask)
    eligible = {
        a: set("F_" + FS.summary[(FS.summary.underlying == a) & FS.summary.eligible].family)
        for a in CONTRACT_ASSETS
    }
    DIR = {a: [c for c in FS.directional().columns if c in eligible.get(a, set())] for a in CONTRACT_ASSETS}
    STATE = {a: [c for c in FS.state().columns if c in eligible.get(a, set())] for a in CONTRACT_ASSETS}

    bundles = {}
    for asset, g0 in df.groupby("underlying"):
        idx = g0.index
        g = g0.reset_index(drop=True)
        E = FS.factors.loc[idx, DIR[asset]].reset_index(drop=True)
        d = build_design(g, f_clock.loc[idx].to_numpy(), extra=E if len(E.columns) else None)
        FF = FS.factors.loc[idx].reset_index(drop=True)
        fn = DIR[asset] + STATE[asset]
        jd = sdf.build_sdf_data(d, FF, fn)
        bundles[asset] = {"design": d, "joint": jd, "DIR": DIR[asset], "STATE": STATE[asset], "factor_names": fn}
    return {"FS": FS, "bundles": bundles, "f_clock": f_clock}

CTX = build_context(TRAIN_DATES)
FS = CTX["FS"]
print("Eligible factor families:")
display(FS.summary.pivot(index=["channel", "family"], columns="underlying", values="eligible"))
print("Loading stability:")
display(FS.summary.pivot(index=["channel", "family"], columns="underlying", values="loading_stability").round(2))
cover = []
for a, b in CTX["bundles"].items():
    jd = b["joint"]
    cover.append({"underlying": a, "m": jd.m, **{n[2:]: float(jd.M[:, j].mean()) for j, n in enumerate(jd.factor_names)}})
print("Usable factor-transition pair coverage:")
display(pd.DataFrame(cover).set_index("underlying").round(2))

Eligible factor families:


underlying                  BTC    XRP
channel     family                    
directional basis          True   True
            book           True   True
            flow           True   True
            skew           True   True
state       activity      False   True
            liquidity      True   True
            option_level  False  False
            volatility     True   True

Loading stability:


underlying                 BTC   XRP
channel     family                  
directional basis          NaN   NaN
            book          1.00  0.99
            flow          1.00  1.00
            skew          1.00  1.00
state       activity      0.69  1.00
            liquidity     1.00  1.00
            option_level  0.71  0.79
            volatility    1.00  1.00

Usable factor-transition pair coverage:


,m,flow,book,basis,skew,volatility,liquidity,activity
underlying,,,,,,,,
BTC,6,1.00,0.94,1.0,0.56,1.0,0.94,NaN
XRP,7,0.97,0.94,1.0,0.58,1.0,0.94,1.0


# Part A — Training-only screening of omega_F

Use expanding blocked folds only inside TRAIN_DATES. To avoid five expensive EM refits per fold, each fold first fits the K=2 omega_F=0 price-switching model once. That fit also estimates state-conditioned factor persistence from the price-defined posterior states. The omega grid is then scanned **causally by filtering only**, asking whether factor-transition evidence available at time t improves the validation-session score of z_{t+1}.

This is a conservative screening stage: omega_F does not alter parameter estimation inside the fold. If omega_F>0 is selected, the final outer-training fit in Part B is then re-estimated jointly at that omega before the untouched holdout is scored. Select the smallest omega within one standard error of the best training-CV price score. omega_F=0 remains the default unless factor dynamics add reproducible price information.

Screening uses fewer iterations/initializations than the final fit; its purpose is hyperparameter navigation, not state interpretation.


In [4]:
def make_spec(K, omega, seed=0, n_init=N_INIT, max_iter=MAX_ITER):
    return sdf.SDFHMMSpec(
        K=K, omega_f=float(omega), max_iter=max_iter, n_init=n_init, seed=seed,
        factor_shrink=FACTOR_SHRINK, factor_scale_shrink=SCALE_SHRINK
    )

def choose_omega(asset, bundle):
    d, jd = bundle["design"], bundle["joint"]
    folds = blocked_folds(sorted(TRAIN_DATES), n_folds=3, min_train_share=0.45)
    sess_scores = {w: {} for w in OMEGAS}
    diagnostics = []
    for fi, (tr_dates, va_dates) in enumerate(folds):
        tr_rows = sdf.rows_from_sessions(d, tr_dates)
        va_rows = sdf.rows_from_sessions(d, va_dates)
        tr = jd.subset(tr_rows)
        va = jd.subset(va_rows)
        sessions = d.frame["session_date"].to_numpy()[va_rows]

        # One EM fit per fold. omega=0 defines the price-switching benchmark
        # and estimates state-conditioned factor dynamics without letting the
        # factor likelihood alter state inference during estimation.
        t0 = time.perf_counter()
        screen_spec = make_spec(2, 0, seed=100 + fi, n_init=SCREEN_N_INIT, max_iter=SCREEN_MAX_ITER)
        res = sdf.fit(tr, screen_spec)
        fit_seconds = time.perf_counter() - t0
        a_sep = float(np.linalg.norm(res.params.factor_a[1] - res.params.factor_a[0]))
        beta_sep = float(abs(res.params.beta_s[1, 1] - res.params.beta_s[0, 1]))

        # Cheap causal omega scan: factor evidence at t updates the filtered
        # state before z_{t+1} is scored. Parameters remain fixed in screening.
        for w in OMEGAS:
            score_spec = make_spec(2, w, seed=100 + fi, n_init=1, max_iter=1)
            sc = sdf.predictive_price_scores(res.params, va, score_spec)
            for sd, value in pd.Series(sc).groupby(sessions).sum().items():
                sess_scores[w][sd] = sess_scores[w].get(sd, 0.0) + float(value)
            diagnostics.append({
                "underlying": asset, "fold": fi, "omega": w,
                "price_score": float(sc.sum()), "a_sep": a_sep,
                "beta_D_sep": beta_sep,
                "fit_seconds": fit_seconds if np.isclose(w, 0) else 0.0,
            })

    S = pd.DataFrame(sess_scores).sort_index()
    total = S.sum()
    best = float(total.idxmax())
    rows = []
    for w in OMEGAS:
        dd = S[w] - S[best]
        se = float(dd.std(ddof=1) * np.sqrt(len(dd))) if len(dd) > 1 else 0.0
        rows.append({
            "omega": w, "cv_price_logscore": float(total[w]),
            "diff_vs_best": float(total[w] - total[best]), "se_diff": se,
            "within_1se": bool(w == best or total[w] >= total[best] - se),
        })
    tab = pd.DataFrame(rows).sort_values("omega")
    chosen = float(tab.loc[tab.within_1se, "omega"].min())
    tab["selected"] = tab.omega == chosen
    return chosen, tab, pd.DataFrame(diagnostics), S

OMEGA_SEL = {}
omega_tabs = []
fold_diag = []
for asset, bundle in CTX["bundles"].items():
    t0 = time.perf_counter()
    w, tab, diag, sess = choose_omega(asset, bundle)
    OMEGA_SEL[asset] = w
    omega_tabs.append(tab.assign(underlying=asset))
    fold_diag.append(diag)
    print(f"{asset}: selected omega_F={w:g} ({time.perf_counter()-t0:.0f}s)")

omega_cv = pd.concat(omega_tabs, ignore_index=True)
fold_dynamics = pd.concat(fold_diag, ignore_index=True)
base = omega_cv[omega_cv.omega == 0].set_index("underlying").cv_price_logscore
omega_cv["vs_omega0"] = omega_cv.cv_price_logscore - omega_cv.underlying.map(base)
print("Training-CV filtering score relative to omega=0:")
display(omega_cv.pivot(index="omega", columns="underlying", values="vs_omega0").round(2))
print("Selected omega_F:", OMEGA_SEL)
sel_diag = fold_dynamics[fold_dynamics.apply(lambda r: np.isclose(r.omega, OMEGA_SEL[r.underlying]), axis=1)]
print("Price-state-conditioned factor persistence separation in the screening fits:")
display(sel_diag.groupby("underlying")[["a_sep", "beta_D_sep"]].agg(["mean", "min", "max"]).round(3))


BTC: selected omega_F=0 (499s)
XRP: selected omega_F=0 (545s)
Training-CV filtering score relative to omega=0:


underlying,BTC,XRP
omega,,
0.00,0.00,0.00
0.10,-0.17,-0.13
0.25,-0.45,-0.40
0.50,-0.97,-1.07
1.00,-2.17,-3.14


Selected omega_F: {'BTC': 0.0, 'XRP': 0.0}
Price-state-conditioned factor persistence separation in the screening fits:


a_sep               beta_D_sep              
             mean    min    max       mean    min    max
underlying                                              
BTC         0.072  0.066  0.079      0.185  0.055  0.275
XRP         0.063  0.046  0.075      0.727  0.469  0.946

# Part B — Final untouched holdout comparison

Refit on all outer training sessions and score the second-half holdout once.

Nested models:
1. K1 — no switching.
2. Price K2 omega0 — same price equation but factor transition evidence does not affect the state.
3. Joint K2 — **fully re-estimated jointly** at the training-selected omega_F when omega_F>0.

If screening selects omega_F=0, the Joint K2 and Price K2 omega0 models are identical and the duplicate expensive fit is skipped.

The primary comparison is Joint K2 versus Price K2 omega0. A factor-dynamic split that does not improve held-out price score is descriptive only.


In [5]:
FINAL = {}
cmp_rows = []
for asset, bundle in CTX["bundles"].items():
    d, jd = bundle["design"], bundle["joint"]
    tr_rows = sdf.rows_from_sessions(d, TRAIN_DATES)
    te_rows = sdf.rows_from_sessions(d, TEST_DATES)
    tr = jd.subset(tr_rows)
    te = jd.subset(te_rows)
    sessions = d.frame["session_date"].to_numpy()[te_rows]
    w = OMEGA_SEL[asset]

    spec_k1 = make_spec(1, 0, seed=1)
    spec_price = make_spec(2, 0, seed=2)
    model_k1 = sdf.fit(tr, spec_k1)
    model_price = sdf.fit(tr, spec_price)

    if np.isclose(w, 0.0):
        spec_joint = spec_price
        model_joint = model_price
    else:
        spec_joint = make_spec(2, w, seed=3)
        model_joint = sdf.fit(tr, spec_joint)

    specs = {"K1": spec_k1, "Price K2 omega0": spec_price, "Joint K2": spec_joint}
    models = {"K1": model_k1, "Price K2 omega0": model_price, "Joint K2": model_joint}
    scores = {name: sdf.predictive_price_scores(res.params, te, specs[name]) for name, res in models.items()}

    pairs = [("Joint K2", "Price K2 omega0"), ("Joint K2", "K1"), ("Price K2 omega0", "K1")]
    cr = []
    for aa, bb in pairs:
        cr.append({"underlying": asset, "model": aa, "vs": bb,
                   **paired_sessions(scores[aa], scores[bb], sessions, n_boot=N_BOOT)})
    cmp_rows.append(pd.DataFrame(cr))
    FINAL[asset] = {"train": tr, "test": te, "sessions": sessions, "models": models, "specs": specs, "scores": scores}
    print(f"{asset}: omega={w:g} | train {tr.T} | test {te.T}")

final_cmp = pd.concat(cmp_rows, ignore_index=True)
SHOW = ["underlying", "model", "vs", "n_sessions", "mean_session_diff", "boot_lo", "boot_hi",
        "median_session_diff", "sessions_won", "loso_min", "loso_max", "per_bar_x1000", "per_bar_nw_t"]
display(final_cmp[SHOW].round(3))
primary = final_cmp[(final_cmp.model == "Joint K2") & (final_cmp.vs == "Price K2 omega0")].copy()
print("Primary comparison: Joint K2 minus Price K2 omega0")
display(primary[["underlying", "mean_session_diff", "boot_lo", "boot_hi",
                 "sessions_won", "loso_min", "loso_max"]].round(3))


BTC: omega=0 | train 1641 | test 1907
XRP: omega=0 | train 1640 | test 1906


,underlying,model,vs,n_sessions,mean_session_diff,boot_lo,boot_hi,median_session_diff,sessions_won,loso_min,loso_max,per_bar_x1000,per_bar_nw_t
0,BTC,Joint K2,Price K2 omega0,9,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,BTC,Joint K2,K1,9,0.219,-1.134,1.573,0.947,0.556,-0.231,0.610,1.033,0.403
2,BTC,Price K2 omega0,K1,9,0.219,-1.134,1.573,0.947,0.556,-0.231,0.610,1.033,0.403
3,XRP,Joint K2,Price K2 omega0,9,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,XRP,Joint K2,K1,9,1.471,-0.014,3.145,0.986,0.667,0.924,1.809,6.945,1.830
5,XRP,Price K2 omega0,K1,9,1.471,-0.014,3.145,0.986,0.667,0.924,1.809,6.945,1.830


Primary comparison: Joint K2 minus Price K2 omega0


,underlying,mean_session_diff,boot_lo,boot_hi,sessions_won,loso_min,loso_max
0,BTC,0.0,0.0,0.0,0.0,0.0,0.0
3,XRP,0.0,0.0,0.0,0.0,0.0,0.0


# Part C — What changes across the two joint states?

These K=2 states are descriptive, not MR/MOM labels. States are ordered by beta_D. Report price parameters, regime-specific diagonal factor persistence versus the common non-switching persistence, dynamic separation, and posterior-mode episodes. A one- or two-bar state is not a tradable structural regime under v3.2.

In [6]:
state_rows = []
dyn_rows = []
ep_rows = []
for asset, v in FINAL.items():
    res = v["models"]["Joint K2"]
    tr = v["train"]
    state_rows.append(sdf.parameter_table(res, tr).assign(underlying=asset, omega=OMEGA_SEL[asset]))
    p = res.params
    for j, n in enumerate(tr.factor_names):
        dyn_rows.append({
            "underlying": asset, "factor": n[2:] if n.startswith("F_") else n,
            "common_a": p.common_a[j], "a_state0": p.factor_a[0, j], "a_state1": p.factor_a[1, j],
            "delta_a": p.factor_a[1, j] - p.factor_a[0, j],
            "sd_state0": p.factor_sd[0, j], "sd_state1": p.factor_sd[1, j],
        })
    ep = sdf.episodes(res, tr)
    ep_rows.append(
        ep.groupby("state").agg(
            episodes=("length", "size"),
            median_bars=("length", "median"),
            p90_bars=("length", lambda x: np.quantile(x, .9))
        ).assign(underlying=asset).reset_index()
    )

states = pd.concat(state_rows, ignore_index=True)
dynamics = pd.DataFrame(dyn_rows)
episode_tab = pd.concat(ep_rows, ignore_index=True)

price_cols = [c for c in states.columns if c.startswith("b_")]
print("Price/state parameters:")
display(states[["underlying", "omega", "state", "occupancy", "sigma_price", "nu"] + price_cols].round(3))
print("Delta factor persistence a_state1 - a_state0:")
display(dynamics.pivot_table(index="factor", columns="underlying", values="delta_a").round(3))
print("Norm of persistence separation:")
display(dynamics.groupby("underlying").delta_a.apply(lambda x: float(np.linalg.norm(x))).rename("a_separation").to_frame().round(3))
print("Training posterior-mode episodes:")
display(episode_tab.set_index(["underlying", "state"]).round(1))

Price/state parameters:


,underlying,omega,state,occupancy,sigma_price,nu,b_const,b_D,b_F_flow,b_F_book,b_F_basis,b_F_skew
0,BTC,0.0,0,0.847,0.764,5.0,0.004,-0.075,-0.021,-0.000,0.064,-0.089
1,BTC,0.0,1,0.153,1.143,5.0,-0.048,0.050,-0.294,0.142,0.041,0.116
2,XRP,0.0,0,0.777,0.811,10.0,-0.023,-0.103,-0.040,0.043,0.039,0.014
3,XRP,0.0,1,0.223,1.322,10.0,0.041,0.279,0.157,-0.303,0.072,0.069


Delta factor persistence a_state1 - a_state0:


underlying,BTC,XRP
factor,,
activity,NaN,-0.008
basis,-0.024,-0.034
book,-0.045,-0.004
flow,-0.100,0.037
liquidity,0.022,-0.013
skew,0.050,-0.031
volatility,-0.013,-0.020


Norm of persistence separation:


,a_separation
underlying,
BTC,0.126
XRP,0.064


Training posterior-mode episodes:


episodes  median_bars  p90_bars
underlying state                                 
BTC        0            19         67.0     154.0
           1            19          7.0      11.2
XRP        0            49         26.0      59.0
           1            51          3.0       6.0

# Part D — Does factor evidence actually move the filtered state?

If omega_F > 0 earns price score but filtered states barely move from omega_F=0, the gain is likely numerical/noise. Very different states without a price-score gain are descriptive factor clusters, not accepted structural regimes.

In [7]:
agree = []
for asset, v in FINAL.items():
    w = OMEGA_SEL[asset]
    r0 = v["models"]["Price K2 omega0"]
    rj = v["models"]["Joint K2"]
    if w == 0:
        agree.append({"underlying": asset, "omega": w, "mode_agreement": 1.0, "mean_abs_prob_shift": 0.0})
        continue
    agree.append({
        "underlying": asset, "omega": w,
        "mode_agreement": float((r0.filtered.argmax(1) == rj.filtered.argmax(1)).mean()),
        "mean_abs_prob_shift": float(np.mean(np.abs(r0.filtered[:, 1] - rj.filtered[:, 1]))),
    })
state_shift = pd.DataFrame(agree)
display(state_shift.round(3))

,underlying,omega,mode_agreement,mean_abs_prob_shift
0,BTC,0.0,1.0,0.0
1,XRP,0.0,1.0,0.0


# Part E — Pre-registered interpretation rules

| Result | Interpretation |
|---|---|
| Training CV selects omega_F=0 | Factor dynamics do not earn a role in state inference at current sample size. |
| omega_F>0 selected, but Joint K2 fails vs omega_F=0 on untouched sessions | Training evidence did not generalize; retain the price-only benchmark. |
| Joint K2 beats omega_F=0 with session-level robustness | First evidence that factor dynamics contain price-relevant regime information; still exploratory. |
| Joint K2 beats omega_F=0 but not K1 | Factor dynamics help the two-state specification, but switching itself is not established. |
| Stable/recurrent a_R differences plus joint price gain | Proceed to the SDF-HSMM duration test. |
| Large a_R differences without price gain | Descriptive factor regimes only. |
| One/two-bar state | Shock/Q candidate, not a tradable structural regime. |

No Bridge, Q_t, dense A_R, directional-by-state interactions, or end-to-end generative null is introduced here. Those remain gated by v3.2 and by this experiment's outcome.

In [8]:
summary = []
for asset in CONTRACT_ASSETS:
    pr = primary[primary.underlying == asset].iloc[0]
    k1 = final_cmp[(final_cmp.underlying == asset) & (final_cmp.model == "Joint K2") & (final_cmp.vs == "K1")].iloc[0]
    asep = float(np.linalg.norm(dynamics[dynamics.underlying == asset].delta_a))
    summary.append({
        "underlying": asset, "omega_selected": OMEGA_SEL[asset],
        "joint_vs_omega0": pr.mean_session_diff,
        "joint_vs_omega0_lo": pr.boot_lo, "joint_vs_omega0_hi": pr.boot_hi,
        "joint_vs_K1": k1.mean_session_diff,
        "sessions_won_vs_omega0": pr.sessions_won,
        "a_separation": asep,
        "provisional_pass": bool(OMEGA_SEL[asset] > 0 and pr.mean_session_diff > 0 and pr.sessions_won > 0.5),
    })
module10_summary = pd.DataFrame(summary)
display(module10_summary.round(3))
print("provisional_pass is only a navigation flag; approximately 15 sessions remain discovery evidence.")

nb10_report = {
    "factor_summary": FS.summary, "omega_cv": omega_cv, "fold_dynamics": fold_dynamics,
    "final_comparison": final_cmp, "states": states, "factor_dynamics": dynamics,
    "episodes": episode_tab, "state_shift": state_shift, "summary": module10_summary,
}
for k, v in nb10_report.items():
    print(f"  {k:<18} {len(v):>5,} rows")
con.close()
print("DuckDB connection closed.")

,underlying,omega_selected,joint_vs_omega0,joint_vs_omega0_lo,joint_vs_omega0_hi,joint_vs_K1,sessions_won_vs_omega0,a_separation,provisional_pass
0,BTC,0.0,0.0,0.0,0.0,0.219,0.0,0.126,False
1,XRP,0.0,0.0,0.0,0.0,1.471,0.0,0.064,False


provisional_pass is only a navigation flag; approximately 15 sessions remain discovery evidence.
  factor_summary        16 rows
  omega_cv              10 rows
  fold_dynamics         30 rows
  final_comparison       6 rows
  states                 4 rows
  factor_dynamics       13 rows
  episodes               4 rows
  state_shift            2 rows
  summary                2 rows
DuckDB connection closed.
